# Chapter 1 — Background: So sánh framework Deep Learning (Chainer / PyTorch / TensorFlow-Keras)

**Bài toán chung của chapter:** Cả 6 file đều giải cùng một bài toán nền tảng — **phân loại ảnh chữ số viết tay MNIST (10 lớp, 0–9)** bằng kiến trúc CNN kiểu **LeNet-5** (Conv → AvgPool → Conv → AvgPool → Conv → Flatten → Dense → Dense). Mục đích của chapter là dùng **cùng một bài toán, cùng một kiến trúc** để so sánh cách 3 framework (Chainer, PyTorch, TensorFlow/Keras) biểu diễn model và vòng lặp huấn luyện khác nhau như thế nào.

| File | Framework | Bài toán cụ thể | Mức độ hoàn chỉnh |
|---|---|---|---|
| `01_chainer.py` | Chainer | Định nghĩa MLP 3 lớp + huấn luyện bằng `Trainer` trên MNIST | Đầy đủ (data → model → train) |
| `03_pytorch.py` | PyTorch | Định nghĩa CNN LeNet-5 bằng subclass `nn.Module` | Chỉ định nghĩa model |
| `04_pytorch.py` | PyTorch | Cùng kiến trúc LeNet-5 nhưng viết gọn bằng `nn.Sequential` | Chỉ định nghĩa model |
| `05_tfkeras.py` | TensorFlow/Keras | Cùng kiến trúc LeNet-5 bằng Keras `Sequential` | Chỉ định nghĩa model |
| `06_fit.py` | TensorFlow/Keras | Kiến trúc LeNet-5 + nạp dữ liệu MNIST thật + `compile`/`fit` | Đầy đủ (data → model → train) |
| `07_training.py` | PyTorch | Kiến trúc LeNet-5 + `DataLoader` + training loop tự viết tay | Đầy đủ (data → model → train loop thủ công) |


## 1. Chainer: MLP + Trainer

### Bài toán
Phân loại 10 chữ số MNIST bằng một **Multilayer Perceptron (MLP)** 3 lớp fully-connected, huấn luyện bằng cơ chế `Trainer` có sẵn của Chainer.

### Giải thích code
1. **Nạp dữ liệu:** `mnist.get_mnist()` trả về `(train, test)` — mỗi phần tử là cặp `(ảnh, nhãn)` đã được Chainer chuẩn hoá sẵn (flatten 28×28 → vector 784, giá trị pixel scale về [0,1]).
2. **Iterator:** `iterators.SerialIterator(train, batchsize=128)` đóng vai trò tương tự `DataLoader` của PyTorch — chia dữ liệu train thành các batch 128 mẫu, dùng để cấp dữ liệu tuần tự cho quá trình huấn luyện.
3. **Kiến trúc MLP (class `MLP(Chain)`):**
   - `l1`: `Linear(None, 100)` — lớp ẩn 1, `None` nghĩa là Chainer sẽ tự suy ra số chiều input từ dữ liệu đầu vào ở lần forward đầu tiên (tương tự lazy init).
   - `l2`: `Linear(None, 100)` — lớp ẩn 2, cùng 100 unit.
   - `l3`: `Linear(None, 10)` — lớp output, 10 unit tương ứng 10 lớp (chữ số 0–9).
   - `forward()`: áp dụng `ReLU` sau `l1` và `l2`, lớp `l3` không có activation (logits thô) vì `Classifier` bên dưới sẽ tự áp `softmax` khi tính loss.
4. **Model:** `L.Classifier(model)` — wrapper tự động thêm hàm mất mát `softmax_cross_entropy` và tính accuracy.
5. **Optimizer:** `MomentumSGD()`, `optimizer.setup(model)`.
6. **Updater + Trainer:** `StandardUpdater` kết hợp `train_iter` và `optimizer` thành 1 bước cập nhật; `Trainer` điều phối toàn bộ vòng lặp huấn luyện qua `max_epoch=10` epoch, log kết quả vào thư mục `mnist_result`.

In [3]:
import chainer
import chainer.functions as F
import chainer.links as L
from chainer import iterators, optimizers, training, Chain
from chainer.datasets import mnist

MNIST_BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"
mnist._retrieve_mnist_training = lambda: mnist._retrieve_mnist(
    "train.npz",
    [
        MNIST_BASE_URL + "train-images-idx3-ubyte.gz",
        MNIST_BASE_URL + "train-labels-idx1-ubyte.gz",
    ],
 )
mnist._retrieve_mnist_test = lambda: mnist._retrieve_mnist(
    "test.npz",
    [
        MNIST_BASE_URL + "t10k-images-idx3-ubyte.gz",
        MNIST_BASE_URL + "t10k-labels-idx1-ubyte.gz",
    ],
 )

train, test = mnist.get_mnist()
batchsize = 128
max_epoch = 10

train_iter = iterators.SerialIterator(train, batchsize)

class MLP(Chain):
    def __init__(self, n_mid_units=100, n_out=10):
        super(MLP, self).__init__()
        with self.init_scope():
            self.l1 = L.Linear(None, n_mid_units)
            self.l2 = L.Linear(None, n_mid_units)
            self.l3 = L.Linear(None, n_out)

    def forward(self, x):
        h1 = F.relu(self.l1(x))
        h2 = F.relu(self.l2(h1))
        return self.l3(h2)

# create model
model = MLP()
model = L.Classifier(model)  # using softmax cross entropy

# set up optimizer
optimizer = optimizers.MomentumSGD()
optimizer.setup(model)

# connect train iterator and optimizer to an updater
updater = training.updaters.StandardUpdater(train_iter, optimizer)

# set up trainer and run
trainer = training.Trainer(updater, (max_epoch, 'epoch'), out='mnist_result')
trainer.run()


## 2. PyTorch: LeNet-5 dạng subclass `nn.Module`

### Bài toán
Cài đặt lại đúng kiến trúc **LeNet-5** (Yann LeCun, 1998) bằng PyTorch, dùng cách viết "tường minh" — subclass `nn.Module` và tự định nghĩa `forward()`.

| Layer | Phép toán | Output shape (ước tính) |
|---|---|---|
| `conv1` | `Conv2d(1→6, kernel 5×5, stride 1, padding 2)` | 6×28×28 (padding=2 giữ nguyên kích thước không gian) |
| `pool1` | `AvgPool2d(2, stride 2)` | 6×14×14 |
| `conv2` | `Conv2d(6→16, kernel 5×5, padding 0)` | 16×10×10 |
| `pool2` | `AvgPool2d(2, stride 2)` | 16×5×5 |
| `conv3` | `Conv2d(16→120, kernel 5×5, padding 0)` | 120×1×1 |
| `flatten` | `Flatten()` | vector 120 |
| `linear4` | `Linear(120→84)` + `tanh` | vector 84 |
| `linear5` | `Linear(84→10)` | vector 10 (logits) |
| `softmax` | `LogSoftmax(dim=1)` | log-xác suất 10 lớp |


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=(5,5), stride=1, padding=2)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1, padding=0)
        self.flatten = nn.Flatten()
        self.linear4 = nn.Linear(120, 84)
        self.linear5 = nn.Linear(84, 10)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = F.tanh(self.conv1(x))
        x = self.pool1(x)
        x = F.tanh(self.conv2(x))
        x = self.pool2(x)
        x = F.tanh(self.conv3(x))
        x = self.flatten(x)
        x = F.tanh(self.linear4(x))
        x = self.linear5(x)
        return self.softmax(x)

model = Model()


## 3. PyTorch: LeNet-5 dạng `nn.Sequential`

### Bài toán
Cài đặt **với kiến trúc LeNet-5**, nhưng dùng cách viết "khai báo" (declarative) bằng `nn.Sequential`.


In [7]:
import torch
import torch.nn as nn

model = nn.Sequential(
    # assume input 1x28x28
    nn.Conv2d(1, 6, kernel_size=(5,5), stride=1, padding=2),
    nn.Tanh(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0),
    nn.Tanh(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(16, 120, kernel_size=5, stride=1, padding=0),
    nn.Tanh(),
    nn.Flatten(),
    nn.Linear(120, 84),
    nn.Tanh(),
    nn.Linear(84, 10),
    nn.LogSoftmax(dim=1)
)


## 4. TensorFlow/Keras: LeNet-5 dạng `Sequential`

### Bài toán
Cài đặt **cùng kiến trúc LeNet-5** bằng TensorFlow/Keras `Sequential` API nhưng khác biệt về API (Keras dùng tham số `activation="..."` gắn liền layer thay vì layer activation riêng).

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, AveragePooling2D, Flatten

model = Sequential([
    Conv2D(6, (5,5), input_shape=(28,28,1), padding="same", activation="tanh"),
    AveragePooling2D((2,2), strides=2),
    Conv2D(16, (5,5), activation="tanh"),
    AveragePooling2D((2,2), strides=2),
    Conv2D(120, (5,5), activation="tanh"),
    Flatten(),
    Dense(84, activation="tanh"),
    Dense(10, activation="softmax")
])


## 5. TensorFlow/Keras: nạp dữ liệu MNIST + huấn luyện

### Bài toán
Định nghĩa kiến trúc mà **huấn luyện thật** trên bộ dữ liệu MNIST chuẩn (60,000 ảnh train / 10,000 ảnh test), đo accuracy trên tập test sau mỗi epoch.

### Giải thích code
1. **Nạp dữ liệu:** `mnist.load_data()` trả về `(X_train, y_train), (X_test, y_test)` — Keras tự động tải bộ MNIST chuẩn (60k/10k), đã tách sẵn train/test theo đúng benchmark.
2. **One-hot encode nhãn:** `to_categorical(y_train)` chuyển nhãn dạng số nguyên (0–9) thành vector one-hot 10 chiều.
3. **`model.compile(...)`:** khai báo `loss="categorical_crossentropy"`, `optimizer="adam"`, theo dõi `metrics=["accuracy"]`.
4. **`model.fit(...)`:** huấn luyện `epochs=100`, `batch_size=32`, đồng thời đánh giá trên `validation_data=(X_test, y_test)` sau mỗi epoch — cho phép theo dõi overfitting qua khoảng cách giữa train loss và validation loss.

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, AveragePooling2D, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

(X_train, y_train), (X_test, y_test) = mnist.load_data()
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

model = Sequential([
    Conv2D(6, (5,5), input_shape=(28,28,1), padding="same", activation="tanh"),
    AveragePooling2D((2,2), strides=2),
    Conv2D(16, (5,5), activation="tanh"),
    AveragePooling2D((2,2), strides=2),
    Conv2D(120, (5,5), activation="tanh"),
    Flatten(),
    Dense(84, activation="tanh"),
    Dense(10, activation="softmax")
])

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, batch_size=32)


2026-09-02 11:37:58.065256: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-02 11:37:58.104831: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-02 11:37:58.104870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-02 11:37:58.106397: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-02 11:37:58.116508: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-02 11:37:58.159789: I tensorflow/core/platform/cpu_feature_guard.cc:1

11490434/11490434 [==============================] - 2s 0us/step
Epoch 1/100


2026-09-02 11:38:03.249704: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 47040000 exceeds 10% of free system memory.


  16/1875 [..............................] - ETA: 20s - loss: 1.6534 - accuracy: 0.4668

2026-09-02 11:38:04.212794: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 24840000 exceeds 10% of free system memory.
2026-09-02 11:38:04.212855: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 24840000 exceeds 10% of free system memory.
2026-09-02 11:38:04.218477: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 25244800 exceeds 10% of free system memory.
2026-09-02 11:38:04.233728: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 24840000 exceeds 10% of free system memory.


1875/1875 [==============================] - 26s 13ms/step - loss: 0.1583 - accuracy: 0.9522 - val_loss: 0.0698 - val_accuracy: 0.9765
Epoch 2/100
1875/1875 [==============================] - 24s 13ms/step - loss: 0.0649 - accuracy: 0.9797 - val_loss: 0.0527 - val_accuracy: 0.9839
Epoch 3/100
1875/1875 [==============================] - 24s 13ms/step - loss: 0.0503 - accuracy: 0.9842 - val_loss: 0.0471 - val_accuracy: 0.9843
Epoch 4/100
1875/1875 [==============================] - 25s 13ms/step - loss: 0.0394 - accuracy: 0.9879 - val_loss: 0.0401 - val_accuracy: 0.9880
Epoch 5/100
1875/1875 [==============================] - 24s 13ms/step - loss: 0.0328 - accuracy: 0.9892 - val_loss: 0.0381 - val_accuracy: 0.9886
Epoch 6/100
1875/1875 [==============================] - 25s 13ms/step - loss: 0.0279 - accuracy: 0.9909 - val_loss: 0.0499 - val_accuracy: 0.9852
Epoch 7/100
1875/1875 [==============================] - 26s 14ms/step - loss: 0.0244 - accuracy: 0.9920 - val_loss: 0.0445 - val_

## 6. PyTorch: `DataLoader` + training loop

### Bài toán
Phiên bản PyTorch "đầy đủ nhất" của chapter: không dùng API huấn luyện có sẵn (như `model.fit()` của Keras) mà **tự viết vòng lặp huấn luyện thủ công** (`training_loop`).

### Giải thích code
1. **Tiền xử lý dữ liệu:** `transforms.Compose([ToTensor()])` chuyển ảnh PIL sang tensor PyTorch, tự động scale pixel về [0,1].
2. **Tải dữ liệu MNIST:** dùng `torchvision.datasets.MNIST(...)`, tự động tải về thư mục `./datafiles/` nếu chưa có.
3. **Chuẩn bị tensor thủ công** (`X_train`, `y_train`, `X_test`, `y_test`): reshape về `(-1, 1, 28, 28)` — định dạng NCHW chuẩn của PyTorch. Phần này thực ra **không được sử dụng** trong phần training loop bên dưới (vốn dùng `train_loader`/`test_loader`).
4. **`DataLoader`:** đóng gói dữ liệu thành các batch 32 mẫu, `shuffle=True` xáo trộn thứ tự mỗi epoch.
5. **Model:** dùng kiến trúc LeNet-5 (`nn.Sequential`).
6. **Hàm `training_loop`:**
   - Vòng lặp `epoch` ngoài cùng (mặc định 100 epoch).
   - **Pha train:** `model.train()` bật chế độ training (ảnh hưởng Dropout/BatchNorm nếu có), forward → tính loss → `zero_grad()` → `backward()` → `optimizer.step()`.
   - **Pha eval:** `model.eval()` tắt chế độ training, tính loss trên `val_loader` nhưng **không có `torch.no_grad()`**.
   - In log gồm epoch, train loss trung bình, val loss trung bình (nếu có `val_loader`).
7. **Optimizer/Loss:** `Adam` + `NLLLoss`.

In [10]:
import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

# Load MNIST data
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(), # required, otherwise MNIST are in PIL format
    #torchvision.transforms.Normalize((0.5,), (0.5,)),
])
train = torchvision.datasets.MNIST('./datafiles/', train=True, download=True, transform=transform)
test = torchvision.datasets.MNIST('./datafiles/', train=True, download=True, transform=transform)

# For manual feed into the model
X_train = train.data.reshape(-1,1,28,28)
y_train = train.targets
X_test = test.data.reshape(-1,1,28,28)
y_test = test.targets

# As iterator for data and target
train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=32, shuffle=True)

# Neural network model
model = nn.Sequential(
    # assume input 1x28x28
    nn.Conv2d(1, 6, kernel_size=(5,5), stride=1, padding=2),
    nn.Tanh(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0),
    nn.Tanh(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(16, 120, kernel_size=5, stride=1, padding=0),
    nn.Tanh(),
    nn.Flatten(),
    nn.Linear(120, 84),
    nn.Tanh(),
    nn.Linear(84, 10),
    nn.LogSoftmax(dim=1)
)

# self-defined training loop function
def training_loop(model, optimizer, loss_fn, train_loader, val_loader=None, n_epochs=100):
    best_loss, best_epoch = np.inf, -1
    best_state = model.state_dict()

    for epoch in range(n_epochs):
        # Training
        model.train()
        train_loss = 0
        for data, target in train_loader:
            output = model(data)
            loss = loss_fn(output, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        # Validation
        model.eval()
        status = (f"{str(datetime.datetime.now())} End of epoch {epoch}, "
                  f"training loss={train_loss/len(train_loader)}")
        if val_loader:
            val_loss = 0
            for data, target in val_loader:
                output = model(data)
                loss = loss_fn(output, target)
                val_loss += loss.item()
            status += f", validation loss={val_loss/len(val_loader)}"
        print(status)

optimizer = optim.Adam(model.parameters())
criterion = nn.NLLLoss()
training_loop(model, optimizer, criterion, train_loader, test_loader, n_epochs=100)


Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./datafiles/MNIST/raw/train-images-idx3-ubyte.gz to ./datafiles/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./datafiles/MNIST/raw/train-labels-idx1-ubyte.gz to ./datafiles/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./datafiles/MNIST/raw/t10k-images-idx3-ubyte.gz to ./datafiles/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./datafiles/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./datafiles/MNIST/raw

2026-09-02 12:22:03.883001 End of epoch 0, training loss=0.23808658559421697, validation loss=0.09938423401812713
2026-09-02 12:22:34.262085 End of epoch 1, training loss=0.07970071202144027, validation loss=0.062326284408072634
2026-09-02 12:23:04.506003 End of epoch 2, training loss=0.05730090724642699, validation loss=0.05224065673214694
2026-09-02 12:23:34.834582 End of epoch 3, training loss=0.04369383990528683, validation loss=0.031423779809086894
2026-09-02 12:24:05.061909 End of epoch 4, training loss=0.035166049275073843, validation loss=0.026148991228050242
2026-09-02 12:24:35.247198 End of epoch 5, training loss=0.02985056490007167, validation loss=0.021940612660651095
2026-09-02 12:25:05.647903 End of epoch 6, training loss=0.024246336878137664, validation loss=0.017414162949104018
2026-09-02 12:25:35.876824 End of epoch 7, training loss=0.02138781471810071, validation loss=0.01799635793907